### Classes EvalGame

Vous trouverez ici la dernière version de classe `Eval`, prenant en compte l'attribut `data` de Game. 

In [ ]:
import os
import pandas as pd


class EvalGame:

    def __init__(self, initialAmount=100):
        self.game = None
        self.initialAmount = initialAmount
        self.results = []
    
    def setGame(self, serie, data, strategy):
        account = Account(self.initialAmount)
        self.game = Game(serie, account, strategy, data=data)

    def eval(self, serie, data, strategy, iteration=10, debug=False):
        ''' 
        eval serie on strategy

        @params serie: serie to assess
        @params data: data used for prediction
        @params strategy : strategy to assess
        @params iteration: number of iteration for random strategy

        @return (gain , cashInvest, tradingFees): gain for the strategy on the whole serie, the amount invested, and the trading fees
        
        '''        
        self.results = []
        self.cashInvest = []
        self.tradingFees = []

        realIteration =1
        if(strategy.isRandom()):
            realIteration = iteration 

        for i in range(0, realIteration):
            self.setGame(serie, data, strategy)
            
            if(debug):
                print("Game:", self.game)
            
            self.game.run(debug=debug)

            score = self.game.getTheoricalGain()
            self.results.append(score)
            self.cashInvest.append(self.game.account.cashInvest)
            self.tradingFees.append(self.game.account.tradingFees)

            if debug:
                print("score[", i, "]: ", score) 

        return self.results, self.cashInvest, self.tradingFees

    def evalTime(self, serie, data, strategy, iteration=10, debug=False):
        ''' 
        eval serie on strategy for each date of the serie

        @params serie: serie to assess
        @params data: data used for prediction
        @params strategy : strategy to assess
        @params iteration: number of iteration for strategy

        @return results: list of gains, one for each start date 
        
        '''     
        results = []
        cashInvests = []
        tradingFees = []

        for t in serie.index: # range(0, serie.shape[0]-1):
            serieT = serie.loc[t:]

            if(data is not None):
                dataT = data.loc[t:]
            else:
                dataT = serie.loc[t:]

            if debug:
                print("serieT:", serieT.shape, " first date:", serieT.index[-1])

            (gain, cashInvest, tradingFee) = self.eval(serieT, dataT, strategy, iteration=iteration, debug=False)
            results.append(np.mean(gain))
            cashInvests.append(np.mean(cashInvest))
            tradingFees.append(np.mean(tradingFee))

        return pd.DataFrame(data={'gain':results, 'cashInvest':cashInvests, 'tradingFees':tradingFees}, index=serie.index)

    def evalSeries(self, series, datas, strategy, iteration=10, debug=False):
        '''
        Eval a list of series on strategy. Series are not supposed to have the same length.

        @params series: list of series
        @params datas: list of dataframes
        @params strategy : strategy to test on the series

        @return list of dataframe, obtained with evalTime()
        '''
        results = []

        if datas is None:
            datas = [None] * len(series)

        for i in range(0, len(series)):
            results.append(self.evalTime(series[i], datas[i], strategy, iteration, debug))
        
        return results
    
    def evalStrategies(self, serie, data, strategies, iteration=10, debug=False):
        '''
        Eval a list of strategies on a serie.

        @params serie: serie
        @params data: data used for prediction
        @params strategies: list of strategies to test the serie on

        @return a tuple (gains, cashInvests), where gains is a DataFrame, 
        with one gain column per Strategy, and cashInvests is DataFrame with one 
        cashInvest column per Strategy
        '''

        gains = pd.DataFrame()
        cashInvests = pd.DataFrame()
        tradingFees = pd.DataFrame()

        for strategy in strategies:
            result = self.evalTime(serie, data, strategy, iteration, debug)

            gains[strategy.getName()] = result.gain
            cashInvests[strategy.getName()] = result.cashInvest
            tradingFees[strategy.getName()] = result.tradingFees

        return gains, cashInvests, tradingFees

    def getResults(self):
        return self.results
    
    @staticmethod
    def saveResults(results, strategyName, serieName, basePath="ResultTest", baseName="EvalTime"):
        ''' 
        Save results to a csv file

        @params results: DataFrame to save
        @params strategyName: name of the strategy used to generate the results
        @params serieName: name of the serie used to generate the results
        @params baseName: base name of the file (default: "EvalTime")
        '''
        if not os.path.exists(basePath):
            os.makedirs(basePath)

        fileToSave =  os.path.join(basePath, baseName + "_" + strategyName + "_" + serieName + ".csv")        
        results.to_csv(fileToSave)

    @staticmethod
    def loadResults(strategyName, serieName, basePath="ResultTest", baseName="EvalTime"):        
        ''' 
        Load results from a csv file

        @params filePath: path to the csv file
        @params strategyName: name of the strategy used to generate the results
        @params serieName: name of the serie used to generate the results
        @params baseName: base name of the file (default: "EvalTime")

        @return results DataFrame
        '''
        fileToLoad = os.path.join(basePath, baseName + "_" + strategyName + "_" + serieName + ".csv")        
        result = pd.read_csv(fileToLoad, index_col=0, parse_dates=True)
        #result.index = pd.to_datetime(result.index)
        return result